
This notebook takes BT-Settl models downloaded from http://svo2.cab.inta-csic.es/theory/newov2/index.php?models=bt-settl and does a few things with them:

(1) Convert from air wavelengths to vacuum wavelengths

(2) Resamples the models onto a standard wavelength grid, using a flux conserving algorithm (Spectres)

(3) Uses bilinear interpolation to place them on the same (logt, logg) grid used by the other stellar libraries in FSPS.

(4) Converts the units from per-unit wavelength to per-unit frequency [erg/s/cm2/angstrom -> erg/s/cm2/Hz/sr]
    
(5) Converts them into the same binary format that FSPS uses for all of its other stellar libraries.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
import tqdm
import re
import os
import glob
import spectres
from astropy import units as u

%matplotlib inline

In [45]:
SPS_HOME = os.path.abspath(os.path.join(os.getcwd(), '..'))
# SPS_HOME = os.getenv('SPS_HOME')
# SPS_HOME = SPS_HOME.replace('fsps', 'fsps_dev')  # -> I call my development folder for fsps 'fsps_dev' to keep it separate from my working fsps installation

# choose one metallicity value for this run
logzi = -1.0
zstr = 'solar' if logzi == 0. else 'halo' if logzi == -1. else 'BAD'

print(SPS_HOME)
print(zstr)

/Users/mreefe/Dropbox/Astrophysics/fsps_dev
halo


In [46]:
# Define the teff, logg, and logz arrays that cover the grid of PAGB models

teff = np.arange(50000, 200000, 10000)
logt = np.log10(teff)
logg = np.arange(5., 9., 1.)
logz = np.array([0.1, 1.])

print(logt)
print(logg)
print(logz)

[4.69897    4.77815125 4.84509804 4.90308999 4.95424251 5.
 5.04139269 5.07918125 5.11394335 5.14612804 5.17609126 5.20411998
 5.23044892 5.25527251 5.2787536 ]
[5. 6. 7. 8.]
[0.1 1. ]


In [47]:
# Define the wavelength grid that we will resample onto
# -> even logarithmic spacing by ~1% of the current wavelength
w_1 = 90. * 1.01**np.arange(0, int(np.log(600/90)/np.log(1.01))) 
# -> finer sampling in the UV/optical; regions of interest
w_2 = np.arange(600., 1800., 0.1)
w_3 = np.arange(1800., 2000., 0.2)
w_4 = np.arange(2000., 3000., 0.5)
w_5 = np.arange(3000., 9000., 1.)
# -> spacing by ~0.3% of the current wavelength
w_6 = 9000. * 1.003**np.arange(0, int(np.log(9.99e6/9000)/np.log(1.003))) 
wavelength = np.concatenate((w_1, w_2, w_3, w_4, w_5, w_6))

print('length = ', len(wavelength))
print(wavelength)

length =  23530
[9.00000000e+01 9.09000000e+01 9.18090000e+01 ... 9.87467880e+06
 9.90430284e+06 9.93401575e+06]


In [48]:
# wavelength must be in angstroms!
def airtovac(wavelength):
    # see: https://www.astro.uu.se/valdwiki/Air-to-vacuum%20conversion
    s = 1e4 / wavelength
    n = 1 + 0.00008336624212083 + 0.02408926869968 / (130.1065924522 - s**2) + 0.0001599740894897 / (38.92568793293 - s**2)
    # do not alter wavelengths below 2000 angstroms 
    wh = np.where(wavelength < 2000.)[0]
    n[wh] = 1.0
    return wavelength * n

In [49]:
c_ang = 299792458e10

# allocate a buffer for all of the spectra at one metallicity
library_in = np.zeros((len(wavelength), len(logt), len(logg)))

folder = f'/Users/mreefe/Dropbox/Astrophysics/stellar_templates/Rauch_PAGB/{zstr}/'
files = glob.glob(os.path.join(folder, f'*_iron'))

for fpath in tqdm.tqdm(files):

    # parse the file name to get the temp, logg, and logz
    fname = os.path.basename(fpath)
    m = re.search(r'^([0-9]+)\_([0-9]+)\_(.+?)\_iron$', fname)
    teff_v = int(m.group(1))
    logg_v = float(m.group(2))/10
    zstr_v = m.group(3)

    # find the indices corresponding to these values in the array
    logt_i = np.where(teff == teff_v)[0][0]
    logg_i = np.where(logg == logg_v)[0][0]

    # print(f'Teff = {teff_v:.0f} (index = {logt_i:.0f})')
    # print(f'logg = {logg_v:.1f} (index = {logg_i:.0f})')
    # print(f'logZ = {logz_v:.1f}')

    # read in the text file
    wave_i, flux_i = np.loadtxt(fpath, unpack=True, comments='*', skiprows=100)

    # Convert the units
    flux_i *= (wave_i * wave_i * 1e-8) / c_ang    # <= convert to erg/s/cm2/Hz
    flux_i *= 1/(4*np.pi)              # <= convert to Harvard flux
    norm = np.trapz(flux_i*c_ang/wave_i**2, wave_i)
    if norm > 0:
        flux_i /= norm

    # perform the flux-conserving resampling onto the output wavelength grid
    flux_o = spectres.spectres(wavelength, wave_i, flux_i, fill=0., verbose=False)
    # flux_o = np.interp(wavelength, wave_i, flux_i, left=0., right=0.)

    # insert it into the 3D array
    library_in[:, logt_i, logg_i] = flux_o

    # # plot the new and old spectrum to compare them
    # fig, ax = plt.subplots()
    # ax.plot(wave_i, flux_i)
    # ax.plot(wavelength, flux_o)
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlabel('Wavelength (angstroms)')
    # ax.set_ylabel('Flambda')
    # # ax.set_xlim(900, 1800)
    # # ax.set_ylim(flux_i[(wave_i > 900) & (wave_i < 1800)].min()*0.98, flux_i[(wave_i > 900) & (wave_i < 1800)].max()*1.02)
    # plt.show()
    # plt.close()


100%|██████████| 51/51 [00:08<00:00,  6.06it/s]


In [50]:
# Read in WMBasic templates for comparison
wmb_logt = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/ipagb.teff'))
wmb_logt = np.log10(wmb_logt)
wmb_logg = np.array([7.0])
wmb = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/ipagb_solar.spec'))
w_wmb = wmb[:,0]
wmb = wmb[:,1:]
wmb = wmb.reshape(wmb.shape[0], len(wmb_logt))

template_folder = os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/Rauch_PAGB/template_plots')
if not os.path.exists(template_folder):
    os.makedirs(template_folder)

for j in range(len(wmb_logg)):
    for i in range(len(wmb_logt)):
        jj = np.nanargmin(np.abs(wmb_logg[j] - logg))
        ii = np.nanargmin(np.abs(wmb_logt[i] - logt))
        fig, ax = plt.subplots()
        ax.plot(w_wmb, wmb[:,i], label='FSPS')
        ax.plot(wavelength, library_in[:,ii,jj], label='Full Res.')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel(r'Wavelength ($\mathring{\rm A}}$)')
        ax.set_ylabel(r'Flux moment (erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$)')
        ax.set_xlim(900, 1800)
        ax.set_ylim(1e-18, 1e-15)
        ax.set_title(f'logg={wmb_logg[j]}, teff={wmb_logt[i]}')
        ax.legend()
        plt.savefig(os.path.join(template_folder, f'{j}_{i}.pdf'), dpi=300, bbox_inches='tight')
        plt.close()

In [51]:
library_in.shape

(23530, 15, 4)

In [52]:
# Save as a text file with the same format as the format 
# need to flatten the array to 2D matching the shape of the WM-Basic arrays FSPS uses
# (flatten logt and logg axis, should be ordered as [t1_g1 t2_g1 t3_g1 ... t1_g2 t2_g2 t3_g2 ...])
library_out = library_in.reshape(library_in.shape[0], len(logg)*len(logt))
# append the wavelength column
library_out = np.concatenate((wavelength.reshape(len(wavelength), 1), library_out), axis=1)

np.savetxt(os.path.join(SPS_HOME, f'SPECTRA/Hot_spectra/Rauch_PAGB/ipagb_{zstr}.spec'), library_out, fmt='%.4e', delimiter=' ')

In [53]:
# save other ancillary files
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/Rauch_PAGB/ipagb.teff'), 10**logt, fmt='%13.1f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/Rauch_PAGB/ipagb.logg'), logg, fmt='%13.5f')